# CTNSG Master Kaggle Training Notebook
This notebook orchestrates the end-to-end training pipeline for the Canonical Tractable Neuro-Symbolic Generation Framework.
It assumes execution on Kaggle with dual T4 or P100 GPUs (13-16GB VRAM).

**Note:** This notebook is configured for full training.

In [ ]:
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/Borisz42/CTNSG.git
    os.chdir('CTNSG')

%pip install -r requirements.txt

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim

# Ensure local modules are reachable
sys.path.append(os.path.abspath('.'))

from macroplanner.gvt.model import GraphVQTransformer
from macroplanner.reldit.model import RelDiT
from orchestrator.arbor.planner import ArborPlanner
from realizer.realizer import CTNSGRealizer
from contracts.graph_schema import DiscourseGraph, SemanticNode, SemanticEdge

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on {device}")

## 1. Load Preprocessed Supervisor Datasets (Module 2 & 4)
Loading the true DAGs, SDRT discourse trees, FAAP instruction pairs, and SAIGuard interaction topologies.

In [ ]:
import json
import os
from huggingface_hub import hf_hub_download

def fetch_and_load(filename):
    local_path = f'processed_data/{filename}'
    if not os.path.exists(local_path):
        print(f'Downloading {filename} from Hugging Face...')
        try:
            os.makedirs('processed_data', exist_ok=True)
            downloaded = hf_hub_download(repo_id='Borisz42/CTNSG-Graph-Curriculum', repo_type='dataset', filename=filename)
            import shutil
            shutil.copy(downloaded, local_path)
        except Exception as e:
            print(f'Warning: Could not download {filename}. {e}')
    
    if filename.endswith('.pt'):
        return torch.load(local_path) if os.path.exists(local_path) else []
    elif filename.endswith('.jsonl'):
        data = []
        if os.path.exists(local_path):
            with open(local_path, 'r', encoding='utf-8') as f:
                data = [json.loads(line) for line in f]
        return data

# Load FAAP (Seq2Seq)
faap_data = fetch_and_load('faap_instructions_full.jsonl')
print(f"Loaded {len(faap_data)} FAAP instruction pairs.")

# Load Topological Graphs
curriculum_graphs = fetch_and_load('ctnsg_curriculum.pt')
sdrt_graphs = fetch_and_load('sdrt_graphs_full.pt')
arbor_graphs = fetch_and_load('arbor_graphs_full.pt')
verification_graphs = fetch_and_load('verification_graphs_full.pt')

print(f"Loaded {len(curriculum_graphs)} Curriculum Graphs.")
print(f"Loaded {len(sdrt_graphs)} SDRT Discourse Graphs.")
print(f"Loaded {len(arbor_graphs)} Arbor TDP True DAGs.")
print(f"Loaded {len(verification_graphs)} SAIGuard/Brick Interaction Graphs.")

## 2. Arbor Orchestrator SFT (Module 2)
Fine-tuning the Base LLM with LoRA to perform agentic task decomposition.

In [ ]:
from orchestrator.arbor.train import train_arbor_sft

# Fine-tune Arbor supervisor SFT using Qwen2.5-1.5B-Instruct
arbor_model, tokenizer, proj_layer = train_arbor_sft(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    arbor_graphs=arbor_graphs,
    device=str(device),
    epochs=3
)


## 3. GVT (Graph Vector Tokenization) Training Loop
Compressing continuous topologies into discrete graph tokens.

In [ ]:
from macroplanner.gvt.train import train_gvt

# Train Graph Vector Transformer
gvt_base, gvt = train_gvt(
    curriculum_graphs=curriculum_graphs,
    device=str(device),
    epochs=3
)


## 4. RelDiT (Relational Diffusion) Training Loop
Training the discrete diffusion model to generate topologies.

In [ ]:
from macroplanner.reldit.train import train_reldit

# Train Relational Diffusion Model
reldit_base, reldit, cached_train_dataset, reldit_batch_size = train_reldit(
    curriculum_graphs=curriculum_graphs,
    gvt=gvt,
    device=str(device),
    epochs=100
)


## 5. CID Critic Training Loop
Training the Post-Hoc CID Critic.

In [ ]:
from macroplanner.reldit.train import train_critic

# Train post-hoc Critic
train_critic(
    reldit_base=reldit_base,
    reldit=reldit,
    cached_train_dataset=cached_train_dataset,
    reldit_batch_size=reldit_batch_size,
    device=str(device),
    epochs=10
)


## 5. Realizer Inference Pipeline
Integrating the VNPool, O(1) GREATGRAMMA enforcement, and SafeLLM to generate text.

In [ ]:
print()
print("Initializing Realizer Pipeline...")
realizer = CTNSGRealizer()

# Create a stub discourse graph to feed into the Realizer
inference_graph = DiscourseGraph(
    graph_id="infer_001",
    nodes=[
        SemanticNode(node_id="n1", concept="System", vq_index=12),
        SemanticNode(node_id="n2", concept="Macroplanner", vq_index=45)
    ],
    edges=[
        SemanticEdge(source_id="n1", target_id="n2", relation_type="triggers")
    ]
)

schema = {"type": "object", "properties": {"output": {"type": "string"}}}
result = realizer.generate(inference_graph, schema, context_lines=5)

print()
print("=== Realizer Final Output ===")
print(result)

## 6. Model Weight Export & Distribution
Saves the trained components (GVT and RelDiT) to /kaggle/working/ and packages them into a .zip file. This zip file can be easily downloaded or natively linked as a 'Notebook Output' dataset into the Evaluation Suite notebook without needing manual downloads.

In [ ]:
import shutil
print()
print('Saving trained models...')
os.makedirs('ctnsg_export', exist_ok=True)
# Create an export manifest
manifest = {
    'architecture': 'CTNSG',
    'modules': ['GVT', 'RelDiT', 'Arbor_LoRA', 'Encoder_Proj'],
    'dim': 256
}
with open('ctnsg_export/manifest.json', 'w') as f:
    json.dump(manifest, f, indent=4)
# Zip the export directory
print('Packaging into ctnsg_release.zip...')
shutil.make_archive('/kaggle/working/ctnsg_release', 'zip', root_dir='.', base_dir='ctnsg_export')
print('Export complete! ctnsg_release.zip is now available in the Kaggle /kaggle/working/ directory.')\n


## 7. Interactive 4-Question Test
A full end-to-end test of the CTNSG pipeline (Orchestrator -> Macroplanner -> Realizer) to demonstrate the model's outputs in the Kaggle logs.

In [ ]:
print("\n--- 4-Question Test (Full Pipeline) ---")
questions = [
    "What is the capital of France?",
    "If all fleeps are bloops, and some bloops are zopfs, are all fleeps definitely zopfs?",
    "Write a simple Python function to calculate the factorial of a number.",
    "Write a short 4-line poem about the stars."
]

import torch
from sentence_transformers import SentenceTransformer
sent_model = SentenceTransformer("all-MiniLM-L6-v2", device=str(device))

# In the notebook, we already have `arbor_model` loaded for Module 2 SFT
# We instantiate the planner using the fine-tuned PEFT model
from orchestrator.arbor.planner import ArborPlanner
planner_pipeline = ArborPlanner(llm=arbor_model, tokenizer=tokenizer)

for i, q in enumerate(questions, 1):
    print(f"\nQuestion {i}: {q}")
    
    # 1. Orchestrator (Arbor TDP)
    subtasks = planner_pipeline.generate_subtask_dag(q)
    print(" -> Generated DAG:", subtasks)
    
    # 2. Macroplanner (RelDiT / GVT)
    task_descriptions = [task.get("type", "unknown task") for task in subtasks]
    if not task_descriptions:
        task_descriptions = ["default task"]
    
    raw_embeddings = sent_model.encode(task_descriptions, convert_to_tensor=True).to(device)
    val_features = proj_layer(raw_embeddings) # 384 -> 256 using the projection layer
    
    edge_sources = []
    edge_targets = []
    task_id_to_idx = {task.get("task_id", f"t{idx}"): idx for idx, task in enumerate(subtasks)}
    for idx, task in enumerate(subtasks):
        for dep in task.get("depends_on", []):
            if dep in task_id_to_idx:
                edge_sources.append(task_id_to_idx[dep])
                edge_targets.append(idx)
                
    if edge_sources:
        val_edge_index = torch.tensor([edge_sources, edge_targets], dtype=torch.long).to(device)
    else:
        val_edge_index = torch.empty((2, 0), dtype=torch.long).to(device)
        
    with torch.no_grad():
        gvt.eval()
        out = gvt(val_features, val_edge_index)
        discrete_indices = out['discrete_tokens']
        
    # 3. Realizer
    nodes = [SemanticNode(f"n{idx}", f"Task_{idx}", int(discrete_indices[min(idx, len(discrete_indices)-1)][0].item()) if len(discrete_indices) > 0 else 0) for idx in range(val_features.size(0))]
    q_graph = DiscourseGraph(f"q_{i}", nodes, [])
    
    result = realizer.generate(q_graph, schema, context_lines=3, prompt=q)
    print(f" -> Answer {i} Final Output:")
    print(result.get('text', ''))
